In [ ]:
from pathlib import Path
import polars as pl

ASSETS_DIR = Path("/Users/henry/data/msc_thesis/climate-attitudes-old/")
CONSTRUCTED_ASSETS_DIR = Path(
    "/Users/henry/data/msc_thesis/climate-attitudes-constructed"
)

## Item name / column name map

In [ ]:
name_remap = (
    pl.read_excel(ASSETS_DIR / "Codebook_220528.xlsx")
    .select(
        pl.col("Variable name").alias("codebook_name"),
        pl.col("Column name").alias("item_name_raw"),
    )
    .with_columns(
        pl.col("item_name_raw")
        .str.replace_all(".", "__", literal=True)
        .alias("item_name")
    )
    .unique(maintain_order=True)
)
name_remap.write_csv(CONSTRUCTED_ASSETS_DIR / "variable_names.csv")

## Lee 2025 variables

In [ ]:
(
    pl.read_excel(ASSETS_DIR / "Codebook_220528.xlsx")
    .select(
        pl.col("Column name").alias("name"),
        pl.col(r"^lee_2025_.*$").replace_strict(
            {"T": True, None: False}, return_dtype=pl.Boolean
        ),
    )
    .filter(pl.any_horizontal(pl.col(r"^lee_2025_.*$")))
    .unique(maintain_order=True)
    .join(
        name_remap.drop("codebook_name"),
        left_on="name",
        right_on="item_name_raw",
        how="left",
    )
    .with_columns(pl.col("item_name").alias("name"))
    .drop("item_name")
    .write_csv(CONSTRUCTED_ASSETS_DIR / "lee_2025_items.csv")
)

## Error items

In [ ]:
ERROR_ITEMS = [
    "pol_expectation_trum",
    "pol_expectation_bide",
    "dem_wfh",
    "dem_ui",
]
pl.DataFrame({"name": ERROR_ITEMS}).write_csv(
    CONSTRUCTED_ASSETS_DIR / "error_items.csv"
)

## Ideology type

In [ ]:
(
    pl.read_excel(ASSETS_DIR / "Codebook_220528.xlsx")
    .select(
        pl.col("Column name").alias("name"),
        pl.col("Ideology type").alias("ideology_type").str.split(";"),
    )
    .with_columns(
        pl.col("ideology_type")
        .list.contains("operational")
        .alias("ideology_operational"),
        pl.col("ideology_type").list.contains("symbolic").alias("ideology_symbolic"),
    )
    .select("name", "ideology_operational", "ideology_symbolic")
    .with_columns(pl.col(r"^ideology_.*$").fill_null(False))
    .filter(pl.any_horizontal(r"^ideology_.*$"))
    .unique(maintain_order=True)
    .join(
        name_remap.drop("codebook_name"),
        left_on="name",
        right_on="item_name_raw",
        how="left",
    )
    .with_columns(pl.col("item_name").alias("name"))
    .drop("item_name")
    .write_csv(CONSTRUCTED_ASSETS_DIR / "ideology_type.csv")
)